# Gold Layer – disease_medication_pharmacy_gold

## Purpose
This notebook creates the main Gold analytical table for the healthcare medication availability platform.

## Business Goal
The table is designed to answer the core business question:

**For a given disease, where in the United States can patients find the required medications?**

## Source Tables
The Gold table is built from the following Silver layer tables:
- `capstone.silver.diseases`
- `capstone.silver.medications`
- `capstone.silver.inventory`
- `capstone.silver.pharmacy`

## Output Table
- `capstone.gold.disease_medication_pharmacy_gold`

## Grain
One row in this table represents:

**one disease + one medication + one pharmacy combination using the latest valid inventory record**

## Key Business Capabilities Supported
This table supports:
- disease → medication → pharmacy relationship mapping
- availability analysis by city, county, state, and region
- state-level and region-level reporting
- medication price analysis
- dashboard filtering by disease, medication, city, county, state, and region
- inventory freshness analysis using the latest update timestamp

## Data Quality Rules Applied
The transformation applies the following rules:
- remove records with missing business keys
- exclude invalid quantity values
- exclude negative prices
- keep only the latest inventory record per pharmacy and medication
- remove duplicate output rows

In [0]:
%sql
-- ============================================================
-- Gold Table: disease_medication_pharmacy_gold
-- Purpose:
-- Create the main analytical Gold dataset that links diseases,
-- medications, pharmacies, geography, and inventory availability.
--
-- Business question supported:
-- For a given disease, where can the required medications be
-- found across pharmacies and geographic regions in the U.S.?
--
-- Target table:
-- capstone.gold.disease_medication_pharmacy_gold
-- ============================================================

CREATE OR REPLACE TABLE capstone.gold.disease_medication_pharmacy_gold
USING DELTA
AS

WITH

-- ------------------------------------------------------------
-- Step 1: Select valid disease reference data
-- ------------------------------------------------------------
disease_base AS (
    SELECT DISTINCT
        disease_id,
        disease_name
    FROM capstone.silver.diseases
    WHERE disease_id IS NOT NULL
      AND TRIM(disease_id) <> ''
      AND disease_name IS NOT NULL
      AND TRIM(disease_name) <> ''
),

-- ------------------------------------------------------------
-- Step 2: Select valid medication reference data
-- ------------------------------------------------------------
medication_base AS (
    SELECT DISTINCT
        medication_id,
        medication_name,
        disease_id
    FROM capstone.silver.medications
    WHERE medication_id IS NOT NULL
      AND TRIM(medication_id) <> ''
      AND medication_name IS NOT NULL
      AND TRIM(medication_name) <> ''
      AND disease_id IS NOT NULL
      AND TRIM(disease_id) <> ''
),

-- ------------------------------------------------------------
-- Step 3: Select valid pharmacy reference data
-- Keep location attributes required for filtering and reporting
-- ------------------------------------------------------------
pharmacy_base AS (
    SELECT DISTINCT
        pharmacy_id,
        pharmacy_name,
        city,
        county,
        state,
        region
    FROM capstone.silver.pharmacy
    WHERE pharmacy_id IS NOT NULL
      AND TRIM(pharmacy_id) <> ''
      AND pharmacy_name IS NOT NULL
      AND TRIM(pharmacy_name) <> ''
),

-- ------------------------------------------------------------
-- Step 4: Clean inventory data
-- Apply data quality rules for price, quantity, and status
-- ------------------------------------------------------------
inventory_base AS (
    SELECT
        pharmacy_id,
        medication_id,
        price,
        quantity,
        last_updated_ts AS last_updated,
        is_available,
        record_status
    FROM capstone.silver.inventory
    WHERE pharmacy_id IS NOT NULL
      AND TRIM(pharmacy_id) <> ''
      AND medication_id IS NOT NULL
      AND TRIM(medication_id) <> ''
      AND quantity IS NOT NULL
      AND quantity > 0
      AND (price IS NULL OR price >= 0)
      AND (record_status IS NULL OR LOWER(record_status) <> 'inactive')
      AND (is_available IS NULL OR is_available = true)
),

-- ------------------------------------------------------------
-- Step 5: Keep only the latest inventory record
-- per pharmacy and medication
-- ------------------------------------------------------------
inventory_latest AS (
    SELECT
        pharmacy_id,
        medication_id,
        price,
        quantity,
        last_updated
    FROM (
        SELECT
            pharmacy_id,
            medication_id,
            price,
            quantity,
            last_updated,
            ROW_NUMBER() OVER (
                PARTITION BY pharmacy_id, medication_id
                ORDER BY last_updated DESC
            ) AS rn
        FROM inventory_base
    ) ranked_inventory
    WHERE rn = 1
),

-- ------------------------------------------------------------
-- Step 6: Build final Gold dataset
-- ------------------------------------------------------------
gold_dataset AS (
    SELECT
        d.disease_id,
        d.disease_name,
        m.medication_id,
        m.medication_name,
        p.pharmacy_id,
        p.pharmacy_name,
        p.city,
        p.county,
        p.state,
        p.region,
        i.price,
        i.quantity,
        i.last_updated
    FROM disease_base d
    INNER JOIN medication_base m
        ON d.disease_id = m.disease_id
    INNER JOIN inventory_latest i
        ON m.medication_id = i.medication_id
    INNER JOIN pharmacy_base p
        ON i.pharmacy_id = p.pharmacy_id
)

-- ------------------------------------------------------------
-- Final safeguard against duplicate rows
-- ------------------------------------------------------------
SELECT DISTINCT *
FROM gold_dataset;

In [0]:
%sql
SELECT 'Total Rows' AS check_name, COUNT(*) AS result
FROM capstone.gold.disease_medication_pharmacy_gold

UNION ALL

SELECT 'Missing Disease ID', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE disease_id IS NULL OR TRIM(disease_id) = ''

UNION ALL

SELECT 'Missing Medication ID', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE medication_id IS NULL OR TRIM(medication_id) = ''

UNION ALL

SELECT 'Missing Pharmacy ID', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE pharmacy_id IS NULL OR TRIM(pharmacy_id) = ''

UNION ALL

SELECT 'Missing Disease Name', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE disease_name IS NULL OR TRIM(disease_name) = ''

UNION ALL

SELECT 'Missing Medication Name', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE medication_name IS NULL OR TRIM(medication_name) = ''

UNION ALL

SELECT 'Missing Pharmacy Name', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE pharmacy_name IS NULL OR TRIM(pharmacy_name) = ''

UNION ALL

SELECT 'Missing City', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE city IS NULL OR TRIM(city) = ''

UNION ALL

SELECT 'Missing County', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE county IS NULL OR TRIM(county) = ''

UNION ALL

SELECT 'Missing State', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE state IS NULL OR TRIM(state) = ''

UNION ALL

SELECT 'Missing Region', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE region IS NULL OR TRIM(region) = ''

UNION ALL

SELECT 'Negative Price', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE price < 0

UNION ALL

SELECT 'Non-Positive Quantity', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE quantity IS NULL OR quantity <= 0

UNION ALL

SELECT 'Missing Last Updated', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_gold
WHERE last_updated IS NULL

UNION ALL

SELECT 'Duplicate Business Rows', COUNT(*)
FROM (
    SELECT
        disease_id,
        medication_id,
        pharmacy_id,
        COUNT(*) AS cnt
    FROM capstone.gold.disease_medication_pharmacy_gold
    GROUP BY disease_id, medication_id, pharmacy_id
    HAVING COUNT(*) > 1
) dup;

In [0]:
%sql
SELECT *
FROM capstone.gold.disease_medication_pharmacy_gold
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT disease_id) AS distinct_diseases,
    COUNT(DISTINCT medication_id) AS distinct_medications,
    COUNT(DISTINCT pharmacy_id) AS distinct_pharmacies,
    COUNT(DISTINCT state) AS distinct_states,
    COUNT(DISTINCT region) AS distinct_regions
FROM capstone.gold.disease_medication_pharmacy_gold;